# Assign LD span for SNPs based on LD-decay within genomic windows

In [2]:
rm(list=ls())
library(tidyverse)
library(readxl)

# change these variables as needed
wdir = "/lustre/orion/syb111/proj-shared/Personal/abeyratnecr/switchgrass/sub_proj/snp2geneLD"
# snplist_fn = paste0(wdir, "/switchgrass_vessel_num_SNPs.xlsx")
snplist_fn = paste0(wdir, "/snp2genes_IronHorseFarm_GWASout_with_AT_annotation_cra.txt")
window_list_fn = paste0(wdir, "/pw_ld/1000000_500000.bed") # filename of the bed file with genomic intervals for LD estimation

setwd(wdir)

In [3]:
# gwas_master_ss <- read_xlsx(path=snplist_fn, col_names=TRUE)
gwas_master_ss <- read.table(file=snplist_fn, header=TRUE)

dim(gwas_master_ss)
str(gwas_master_ss)

[1] 23486    12

'data.frame':	23486 obs. of  12 variables:
 $ gwas_model        : chr  "BLINK" "BLINK" "BLINK" "BLINK" ...
 $ trait             : chr  "ihf_total_dry_biomass_kg_2021_1" "ihf_total_dry_biomass_kg_2021_1" "ihf_total_dry_biomass_kg_2021_1" "ihf_shooting_0_3_2020_1" ...
 $ SNP               : chr  "Chr01K_699960" "Chr01K_699960" "Chr01K_699960" "Chr01K_718666" ...
 $ Chr               : chr  "01K" "01K" "01K" "01K" ...
 $ Pos               : int  699960 699960 699960 718666 718666 744204 744204 744204 744204 744204 ...
 $ P.value           : num  2.67e-08 2.67e-08 2.67e-08 1.30e-09 1.30e-09 ...
 $ MAF               : num  0.0964 0.0964 0.0964 0.0693 0.0693 ...
 $ nobs              : int  223 223 223 231 231 231 231 231 231 231 ...
 $ H.B.P.Value       : num  0.001231 0.001231 0.001231 0.000231 0.000231 ...
 $ Effect            : num  -0.85 -0.85 -0.85 0.424 0.424 ...
 $ Adjusted.P.value  : num  0.005914 0.005914 0.005914 0.000179 0.000179 ...
 $ Bonferroni_P.value: num  3.55e-01 3.55e-01 3

In [4]:
# check the min and max p-values of SNPs
gwas_master_ss %>% pull(P.value) %>% min()
gwas_master_ss %>% pull(P.value) %>% max()

gwas_master_ss %>% pull(Bonferroni_P.value) %>% min()
gwas_master_ss %>% pull(Bonferroni_P.value) %>% max()

[1] 4.28e-56

[1] 3.76e-08

[1] 5.69e-49

[1] 0.5

In [5]:
# check the number of unique SNPs across all the models and traits
gwas_master_ss %>% pull(SNP) %>% unique() %>% length()

[1] 5533

In [6]:
# Input window list
windows <- read.table(window_list_fn, header=F)
colnames(windows) <- c('Chr', 'Start', 'End')
# dim(windows)
# head(windows)

# read the model fit and span values for intervals after modelling LD-decay
decay_fit <- NULL
missing_files <- NULL
missing_iterations <- NULL
for (i in 1:nrow(windows)) {
    filename <- paste0(wdir, '/pw_ld/', windows$Chr[i], '_', windows$Start[i], '_', windows$End[i], '.decay')
    if (file.exists(filename)) {
        tmp <- read.table(filename, header=T)
        decay_fit <- rbind(decay_fit, cbind(Chr=windows$Chr[i], Start=windows$Start[i], End=windows$End[i], tmp))
    } else {
        missing_files <- c(missing_files, filename)
        missing_iterations <- c(missing_iterations, i)
    }
}
decay_fit <- decay_fit %>% as.data.frame()
# stopifnot(nrow(decay_fit) == nrow(windows))

dim(decay_fit)
head(decay_fit)

[1] 2205    6

,Chr,Start,End,distance.span,expected.ld,mod_fit
,<chr>,<int>,<int>,<int>,<dbl>,<dbl>
1,Chr01K,0,1000000,9337,0.02307004,0.9549745
2,Chr01K,500000,1500000,7348,0.02111843,0.9777420
3,Chr01K,1000000,2000000,6390,0.01738335,0.9664553
4,Chr01K,1500000,2500000,6266,0.01746169,0.9510637
5,Chr01K,2000000,3000000,6849,0.01807289,0.9592801
6,Chr01K,2500000,3500000,6375,0.01840936,0.9583541


In [7]:
# Are there any missing files for the genomic interval LD estimation?
missing_iterations %>% length()
# At this point check if there are missing LD estimates for genomic windows. Possible reason could be that script-3 did not run to completion

[1] 2

[1] 249 450

In [10]:
# checking just one entry that is missing
i=249
filename <- paste0(wdir, '/pw_ld/', windows$Chr[i], '_', windows$Start[i], '_', windows$End[i], '.decay')
filename

[1] "/lustre/orion/syb111/proj-shared/Personal/abeyratnecr/switchgrass/sub_proj/snp2geneLD/pw_ld/Chr01N_67000000_67001661.decay"

In [8]:
# generate a space separated list that can be re-used with script 4_
for (i in 1:length(missing_iterations)) {
    cat(missing_iterations[i], " ")
}

249  450  

In [10]:
# assigning window spans for each of the SNPs based on overlap with genomic windows
unmatched_idx <- NULL
gwas_with_span <- NULL
for (i in 1:nrow(gwas_master_ss)) {
    tmp_decay_fit <- decay_fit %>% 
    filter(Chr==paste0('Chr', gwas_master_ss$Chr[i])) %>%
    mutate(snp_pos = gwas_master_ss$Pos[i]) %>%
    mutate(olap = ifelse(snp_pos >= Start & snp_pos <= End, 1, 0)) %>% 
    filter(olap == 1)
    # filter(Start <= gwas_master_ss$Pos[i] & End >= gwas_master_ss$Pos[i])

    if (nrow(tmp_decay_fit) == 0) {
        # SNP does not overlap with the window
        unmatched_idx = c(unmatched_idx, i)
        next
    } else if (nrow(tmp_decay_fit) == 1) {
        # SNP overlaps with one window
        span = tmp_decay_fit$distance.span[1]
        mod_fit = tmp_decay_fit$mod_fit[1]
    } else {
        # If SNP overlaps with more than one window taking the average
        span = round(mean(tmp_decay_fit$distance.span),0) 
        mod_fit = mean(tmp_decay_fit$mod_fit)
    }
    
    up_loc = gwas_master_ss$Pos[i] - span
    down_loc = gwas_master_ss$Pos[i] + span
    up_loc = ifelse(up_loc < 0, 0, up_loc)

    gwas_with_span <- rbind(gwas_with_span, cbind(gwas_master_ss[i,],
                                                  mod_fit=mod_fit,
                                                  span=span,
                                                  up_loc=up_loc,
                                                  down_loc=down_loc))
}

In [14]:
dim(gwas_with_span)
# stopifnot(nrow(gwas_master_ss) == nrow(gwas_with_span)) # checking if all SNPs have assigned LD spans
gwas_with_span %>% head()

[1] 23486    16

,gwas_model,trait,SNP,Chr,Pos,P.value,MAF,nobs,H.B.P.Value,Effect,Adjusted.P.value,Bonferroni_P.value,mod_fit,span,up_loc,down_loc
,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,BLINK,ihf_total_dry_biomass_kg_2021_1,Chr01K_699960,01K,699960,2.67e-08,0.09641256,223,1.230638e-03,-0.8497491,5.914254e-03,3.55e-01,0.9663583,8342,691618,708302
2,BLINK,ihf_total_dry_biomass_kg_2021_1,Chr01K_699960,01K,699960,2.67e-08,0.09641256,223,1.230638e-03,-0.8497491,5.914254e-03,3.55e-01,0.9663583,8342,691618,708302
3,BLINK,ihf_total_dry_biomass_kg_2021_1,Chr01K_699960,01K,699960,2.67e-08,0.09641256,223,1.230638e-03,-0.8497491,5.914254e-03,3.55e-01,0.9663583,8342,691618,708302
4,BLINK,ihf_shooting_0_3_2020_1,Chr01K_718666,01K,718666,1.30e-09,0.06926407,231,2.314220e-04,0.4236989,1.793200e-04,1.73e-02,0.9663583,8342,710324,727008
5,BLINK,ihf_shooting_0_3_2020_1,Chr01K_718666,01K,718666,1.30e-09,0.06926407,231,2.314220e-04,0.4236989,1.793200e-04,1.73e-02,0.9663583,8342,710324,727008
6,FarmCPU,ihf_panicle_emergence_2020_1,Chr01K_744204,01K,744204,3.80e-16,0.14069264,231,1.230000e-10,-5.9991126,3.160000e-10,5.05e-09,0.9663583,8342,735862,752546


In [11]:
# check the number of entries without LD span assignment
unmatched_idx %>% length()

# check the number of unique SNPs without LD span assignment
gwas_master_ss[unmatched_idx, ] %>% pull(SNP) %>% unique() %>% length()

[1] 0

In [16]:
# Write the selected hits.
write.csv(gwas_with_span, file=paste0(wdir, '/fsnps/gwas_with_span.csv'), row.names=F, quote=F)

In [19]:
# write the list of unique coordinates to find genes associated with coordinates
query_intervals <- gwas_with_span %>% dplyr::select(Chr, up_loc, down_loc) %>% distinct() %>% arrange(Chr, up_loc)

# Make sure the chromosome names match
query_intervals <- query_intervals %>% mutate(Chr=paste0('Chr', Chr))
write.table(query_intervals, file=paste0(wdir, '/fsnps/query_intervals.bed'), col.names=F, row.names=F, quote=F, sep="\t")

In [ ]:
# END